In [1]:
# Check GPU and CUDA availability.
!nvidia-smi

import torch
import sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
else:
    raise RuntimeError("CUDA is not available. Please switch Colab runtime to GPU.")

Fri May 22 12:28:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Install required packages.
!pip -q install -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" accelerate faiss-cpu tqdm numpy psutil

import importlib.metadata as md
import torch
import faiss

print("transformers:", md.version("transformers"))
print("sentence-transformers:", md.version("sentence-transformers"))
print("faiss:", faiss.__version__)
print("torch:", torch.__version__)

transformers: 5.9.0
sentence-transformers: 5.5.1
faiss: 1.13.2
torch: 2.10.0+cu128


In [3]:
# Mount Google Drive and define project paths.
from google.colab import drive
from pathlib import Path
import shutil
import os
import json

drive.mount("/content/drive")

DATASET = "hotpotqa"

PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
IDEA_DIR = PROJECT_DIR / "idea_1"

if not IDEA_DIR.exists():
    raise FileNotFoundError(
        f"idea_1 folder not found:\n{IDEA_DIR}"
    )

INPUT_FILENAME = "hotpotqa_kg_extractions_all_00000001_to_00035029.clean.json"

# Preferred expected path.
PREFERRED_INPUT_CLEAN_PATH = IDEA_DIR / "kg" / DATASET / INPUT_FILENAME

# If preferred path does not exist, search inside final_project/idea_1.
if PREFERRED_INPUT_CLEAN_PATH.exists():
    INPUT_CLEAN_PATH = PREFERRED_INPUT_CLEAN_PATH
else:
    matches = sorted(IDEA_DIR.rglob(INPUT_FILENAME))

    if not matches:
        raise FileNotFoundError(
            f"Input clean KG file not found under:\n{IDEA_DIR}\n\n"
            f"Searched for filename:\n{INPUT_FILENAME}"
        )

    if len(matches) > 1:
        print("Multiple matching clean files found. Using the first one:")
        for p in matches:
            print(" -", p)

    INPUT_CLEAN_PATH = matches[0]

# Use the folder containing the clean file as the dataset folder.
DATASET_DIR = INPUT_CLEAN_PATH.parent

# Output folder next to the clean file.
CONSTRUCT_DIR = DATASET_DIR / "kg_construct"

LOCAL_WORK_DIR = Path(f"/content/kg_construct_work/{DATASET}")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)
CONSTRUCT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_INPUT_PATH = LOCAL_WORK_DIR / INPUT_CLEAN_PATH.name

# Copy input to local disk for faster reading.
shutil.copy2(INPUT_CLEAN_PATH, LOCAL_INPUT_PATH)

print("Drive input:", INPUT_CLEAN_PATH)
print("Local input:", LOCAL_INPUT_PATH)
print("Drive output folder:", CONSTRUCT_DIR)
print("Local work folder:", LOCAL_WORK_DIR)

Mounted at /content/drive
Drive input: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/hotpotqa_kg_extractions_all_00000001_to_00035029.clean.json
Local input: /content/kg_construct_work/hotpotqa/hotpotqa_kg_extractions_all_00000001_to_00035029.clean.json
Drive output folder: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct
Local work folder: /content/kg_construct_work/hotpotqa


In [4]:
# Define output paths and main configuration.
from pathlib import Path

MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

# Entity embedding settings.
EMBED_BATCH_SIZE = 32
MAX_SEQ_LENGTH = 128
EMBEDDING_DIM = 4096

# Output file names.
NORMALIZED_KG_JSONL_LOCAL = LOCAL_WORK_DIR / f"{DATASET}_kg_extractions.clean.normalized_entities.jsonl"
ENTITY_CATALOG_JSONL_LOCAL = LOCAL_WORK_DIR / f"{DATASET}_entities.jsonl"
ENTITY_EMBEDDINGS_NPY_LOCAL = LOCAL_WORK_DIR / f"{DATASET}_entity_embeddings.float32.npy"
ENTITY_IDS_NPY_LOCAL = LOCAL_WORK_DIR / f"{DATASET}_entity_ids.int64.npy"
ENTITY_FAISS_LOCAL = LOCAL_WORK_DIR / f"{DATASET}_entities.indexflatip.idmap.faiss"
ENTITY_META_JSON_LOCAL = LOCAL_WORK_DIR / f"{DATASET}_entity_index_meta.json"
AUDIT_JSON_LOCAL = LOCAL_WORK_DIR / f"{DATASET}_entity_normalization_audit.json"

print("Model:", MODEL_NAME)
print("Embedding batch size:", EMBED_BATCH_SIZE)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("Expected embedding dim:", EMBEDDING_DIM)
print("Local outputs will be written to:", LOCAL_WORK_DIR)

Model: Qwen/Qwen3-Embedding-8B
Embedding batch size: 32
Max sequence length: 128
Expected embedding dim: 4096
Local outputs will be written to: /content/kg_construct_work/hotpotqa


In [5]:
# Define helpers for safe writing and entity normalization.
import json
import os
import re
import html
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict, OrderedDict

def atomic_json_dump(obj, path):
    # Write JSON safely, then replace the target file.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def normalize_entity_text(text):
    # Normalize entity text while preserving useful punctuation.
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    # Normalize common quote and dash variants.
    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u00a0": " ",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    # Collapse whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    # Remove only surrounding quotes, not internal punctuation.
    text = text.strip(" \t\r\n\"'`")

    # Lowercase as requested.
    text = text.lower()

    return text

def unique_keep_order(items):
    # Deduplicate while preserving first occurrence order.
    seen = set()
    out = []

    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)

    return out

def get_chunk_id(item):
    # Support both old and new key styles.
    return item.get("chunk_id") or item.get("Chunk_id")

def get_title(item):
    # Support both old and new key styles.
    return item.get("title") or item.get("Title")

def get_paragraph_id(item):
    # Support both old and new key styles.
    return item.get("paragraph_id") or item.get("Paragraph_id")

def get_token_count(item):
    # Support both old and new key styles.
    return item.get("token_count") or item.get("Token_count")

In [6]:
# Load clean KG, normalize entity fields, and collect unique entities.
import json
from collections import OrderedDict, Counter, defaultdict
from tqdm.auto import tqdm

with open(LOCAL_INPUT_PATH, "r", encoding="utf-8") as f:
    clean_items = json.load(f)

if not isinstance(clean_items, list):
    raise RuntimeError("Input clean KG file must be a JSON list.")

entity_to_stats = OrderedDict()
normalization_collisions = defaultdict(Counter)

total_relation_edges_in = 0
total_relation_edges_out = 0
total_fact_edges_in = 0
total_fact_edges_out = 0
empty_entity_mentions = 0

def register_entity(normalized_entity, original_entity, source_kind, chunk_id):
    # Register normalized entity and track provenance stats.
    global empty_entity_mentions

    if not normalized_entity:
        empty_entity_mentions += 1
        return

    if normalized_entity not in entity_to_stats:
        entity_to_stats[normalized_entity] = {
            "count": 0,
            "source_counts": Counter(),
            "original_forms": Counter(),
            "example_chunk_ids": [],
        }

    stats = entity_to_stats[normalized_entity]
    stats["count"] += 1
    stats["source_counts"][source_kind] += 1
    stats["original_forms"][str(original_entity)] += 1

    if len(stats["example_chunk_ids"]) < 5 and chunk_id not in stats["example_chunk_ids"]:
        stats["example_chunk_ids"].append(chunk_id)

    normalization_collisions[normalized_entity][str(original_entity)] += 1

with open(NORMALIZED_KG_JSONL_LOCAL, "w", encoding="utf-8") as out_f:
    for input_index, item in enumerate(tqdm(clean_items, desc="Normalizing KG items")):
        chunk_id = get_chunk_id(item)
        title = get_title(item)
        paragraph_id = get_paragraph_id(item)
        token_count = get_token_count(item)

        relations_in = item.get("relations") or []
        facts_in = item.get("facts") or []

        normalized_relations = []
        normalized_facts = []
        item_entities = []

        # Normalize relation heads and tails.
        for rel in relations_in:
            total_relation_edges_in += 1

            if not isinstance(rel, dict):
                continue

            head_original = rel.get("head")
            tail_original = rel.get("tail")
            head_norm = normalize_entity_text(head_original)
            tail_norm = normalize_entity_text(tail_original)

            if not head_norm or not tail_norm:
                empty_entity_mentions += int(not head_norm) + int(not tail_norm)
                continue

            register_entity(head_norm, head_original, "relation_head", chunk_id)
            register_entity(tail_norm, tail_original, "relation_tail", chunk_id)

            item_entities.extend([head_norm, tail_norm])

            new_rel = dict(rel)
            new_rel["head_original"] = head_original
            new_rel["tail_original"] = tail_original
            new_rel["head"] = head_norm
            new_rel["tail"] = tail_norm

            normalized_relations.append(new_rel)
            total_relation_edges_out += 1

        # Normalize fact entities.
        for fact in facts_in:
            total_fact_edges_in += 1

            if not isinstance(fact, dict):
                continue

            entity_original = fact.get("entity")
            entity_norm = normalize_entity_text(entity_original)

            if not entity_norm:
                empty_entity_mentions += 1
                continue

            register_entity(entity_norm, entity_original, "fact_entity", chunk_id)

            item_entities.append(entity_norm)

            new_fact = dict(fact)
            new_fact["entity_original"] = entity_original
            new_fact["entity"] = entity_norm

            normalized_facts.append(new_fact)
            total_fact_edges_out += 1

        normalized_item = {
            "input_index": input_index,
            "chunk_id": chunk_id,
            "title": title,
            "paragraph_id": paragraph_id,
            "token_count": token_count,
            "entities": unique_keep_order(item_entities),
            "relations": normalized_relations,
            "facts": normalized_facts,
            "source_clean_file": str(INPUT_CLEAN_PATH),
        }

        out_f.write(json.dumps(normalized_item, ensure_ascii=False) + "\n")

print("Total clean items:", len(clean_items))
print("Unique normalized entities:", len(entity_to_stats))
print("Relation edges in/out:", total_relation_edges_in, total_relation_edges_out)
print("Fact edges in/out:", total_fact_edges_in, total_fact_edges_out)
print("Empty entity mentions skipped:", empty_entity_mentions)
print("Normalized KG JSONL:", NORMALIZED_KG_JSONL_LOCAL)

Normalizing KG items:   0%|          | 0/35029 [00:00<?, ?it/s]

Total clean items: 35029
Unique normalized entities: 327873
Relation edges in/out: 692402 692402
Fact edges in/out: 559959 559959
Empty entity mentions skipped: 0
Normalized KG JSONL: /content/kg_construct_work/hotpotqa/hotpotqa_kg_extractions.clean.normalized_entities.jsonl


In [7]:
# Save unique entity catalog as JSONL with stable integer IDs.
import json
from tqdm.auto import tqdm

MAX_ORIGINAL_FORMS_PER_ENTITY = 20

entity_records = []
entity_to_id = {}

with open(ENTITY_CATALOG_JSONL_LOCAL, "w", encoding="utf-8") as f:
    for entity_id, (entity, stats) in enumerate(tqdm(entity_to_stats.items(), desc="Writing entity catalog")):
        entity_to_id[entity] = entity_id

        original_forms = [
            {"text": text, "count": count}
            for text, count in stats["original_forms"].most_common(MAX_ORIGINAL_FORMS_PER_ENTITY)
        ]

        record = {
            "entity_id": entity_id,
            "entity": entity,
            "mention_count": stats["count"],
            "source_counts": dict(stats["source_counts"]),
            "original_forms": original_forms,
            "example_chunk_ids": stats["example_chunk_ids"],
        }

        entity_records.append(record)
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

audit = {
    "dataset": DATASET,
    "model_name": MODEL_NAME,
    "input_clean_path": str(INPUT_CLEAN_PATH),
    "normalized_kg_jsonl": str(CONSTRUCT_DIR / NORMALIZED_KG_JSONL_LOCAL.name),
    "entity_catalog_jsonl": str(CONSTRUCT_DIR / ENTITY_CATALOG_JSONL_LOCAL.name),
    "num_clean_items": len(clean_items),
    "num_unique_normalized_entities": len(entity_records),
    "total_relation_edges_in": total_relation_edges_in,
    "total_relation_edges_out": total_relation_edges_out,
    "total_fact_edges_in": total_fact_edges_in,
    "total_fact_edges_out": total_fact_edges_out,
    "empty_entity_mentions_skipped": empty_entity_mentions,
    "normalization": {
        "unicode": "NFKC",
        "lowercase": True,
        "collapse_whitespace": True,
        "preserve_internal_punctuation": True,
    },
}

atomic_json_dump(audit, AUDIT_JSON_LOCAL)

print("Saved entity catalog:", ENTITY_CATALOG_JSONL_LOCAL)
print("Saved audit:", AUDIT_JSON_LOCAL)
print("First 3 entity records:")
print(json.dumps(entity_records[:3], ensure_ascii=False, indent=2))

Writing entity catalog:   0%|          | 0/327873 [00:00<?, ?it/s]

Saved entity catalog: /content/kg_construct_work/hotpotqa/hotpotqa_entities.jsonl
Saved audit: /content/kg_construct_work/hotpotqa/hotpotqa_entity_normalization_audit.json
First 3 entity records:
[
  {
    "entity_id": 0,
    "entity": "meet corliss archer",
    "mention_count": 54,
    "source_counts": {
      "relation_head": 37,
      "fact_entity": 11,
      "relation_tail": 6
    },
    "original_forms": [
      {
        "text": "Meet Corliss Archer",
        "count": 54
      }
    ],
    "example_chunk_ids": [
      "hotpotqa_chunk_00000001",
      "hotpotqa_chunk_00000002",
      "hotpotqa_chunk_00000018",
      "hotpotqa_chunk_00000020",
      "hotpotqa_chunk_00000022"
    ]
  },
  {
    "entity_id": 1,
    "entity": "january 7, 1943",
    "mention_count": 1,
    "source_counts": {
      "relation_tail": 1
    },
    "original_forms": [
      {
        "text": "January 7, 1943",
        "count": 1
      }
    ],
    "example_chunk_ids": [
      "hotpotqa_chunk_00000001"
    ]

In [8]:
# Load Qwen3 embedding model on GPU.
import os
import torch
from sentence_transformers import SentenceTransformer

# Use local cache for faster Colab runtime I/O.
os.environ["HF_HOME"] = str(LOCAL_WORK_DIR / "hf_cache")
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_WORK_DIR / "hf_cache" / "transformers")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda",
    model_kwargs={
        "device_map": "auto",
        "attn_implementation": "sdpa",
    },
    tokenizer_kwargs={
        "padding_side": "left",
    },
)

# Entity names are short, so this speeds up batching and reduces memory.
model.max_seq_length = MAX_SEQ_LENGTH

print("Loaded model:", MODEL_NAME)
print("Max sequence length:", model.max_seq_length)
print("Device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen3-Embedding-8B
Max sequence length: 128
Device: cuda:0


In [9]:
# Encode all entities and save embeddings with model default output dtype.
import numpy as np
from tqdm.auto import tqdm
import torch
import gc

entities = [record["entity"] for record in entity_records]
entity_ids = np.array([record["entity_id"] for record in entity_records], dtype=np.int64)

if len(entities) == 0:
    raise RuntimeError("No entities found. Cannot build embeddings.")

np.save(ENTITY_IDS_NPY_LOCAL, entity_ids)

num_batches = (len(entities) + EMBED_BATCH_SIZE - 1) // EMBED_BATCH_SIZE

embeddings_memmap = None
embedding_dtype = None

for batch_idx in tqdm(range(num_batches), desc="Embedding entities"):
    start = batch_idx * EMBED_BATCH_SIZE
    end = min(start + EMBED_BATCH_SIZE, len(entities))

    batch_texts = entities[start:end]

    batch_embeddings = model.encode(
        batch_texts,
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    batch_embeddings = np.asarray(batch_embeddings)

    if batch_embeddings.shape[1] != EMBEDDING_DIM:
        raise RuntimeError(
            f"Unexpected embedding dim: {batch_embeddings.shape[1]} != {EMBEDDING_DIM}"
        )

    if embeddings_memmap is None:
        embedding_dtype = batch_embeddings.dtype

        embeddings_memmap = np.lib.format.open_memmap(
            ENTITY_EMBEDDINGS_NPY_LOCAL,
            mode="w+",
            dtype=embedding_dtype,
            shape=(len(entities), EMBEDDING_DIM),
        )

        print("Model output embedding dtype:", embedding_dtype)

    if batch_embeddings.dtype != embedding_dtype:
        raise RuntimeError(
            f"Embedding dtype changed during encoding: {batch_embeddings.dtype} != {embedding_dtype}"
        )

    embeddings_memmap[start:end, :] = batch_embeddings

    if batch_idx % 50 == 0:
        embeddings_memmap.flush()

embeddings_memmap.flush()

# Sanity check norms.
sample = np.asarray(embeddings_memmap[: min(1000, len(entities))])
norms = np.linalg.norm(sample, axis=1)

print("Saved embeddings:", ENTITY_EMBEDDINGS_NPY_LOCAL)
print("Saved entity IDs:", ENTITY_IDS_NPY_LOCAL)
print("Embedding shape:", embeddings_memmap.shape)
print("Embedding dtype:", embeddings_memmap.dtype)
print("Norm min/mean/max:", float(norms.min()), float(norms.mean()), float(norms.max()))

del embeddings_memmap
torch.cuda.empty_cache()
gc.collect()

Embedding entities:   0%|          | 0/10247 [00:00<?, ?it/s]

Model output embedding dtype: float32
Saved embeddings: /content/kg_construct_work/hotpotqa/hotpotqa_entity_embeddings.float32.npy
Saved entity IDs: /content/kg_construct_work/hotpotqa/hotpotqa_entity_ids.int64.npy
Embedding shape: (327873, 4096)
Embedding dtype: float32
Norm min/mean/max: 0.9963335990905762 1.001314640045166 1.0038872957229614


262

In [10]:
# Build FAISS cosine-similarity index with explicit entity_id mapping.
import numpy as np
import faiss
from tqdm.auto import tqdm
import json
from datetime import datetime, timezone

embeddings = np.load(ENTITY_EMBEDDINGS_NPY_LOCAL, mmap_mode="r")
entity_ids = np.load(ENTITY_IDS_NPY_LOCAL)

if embeddings.dtype != np.float32:
    raise RuntimeError(f"Embeddings must be float32, got {embeddings.dtype}")

if entity_ids.dtype != np.int64:
    raise RuntimeError(f"Entity IDs must be int64, got {entity_ids.dtype}")

if embeddings.shape[0] != entity_ids.shape[0]:
    raise RuntimeError("Embeddings and entity IDs have different lengths.")

dim = embeddings.shape[1]

base_index = faiss.IndexFlatIP(dim)
index = faiss.IndexIDMap2(base_index)

ADD_BATCH_SIZE = 50000

for start in tqdm(range(0, embeddings.shape[0], ADD_BATCH_SIZE), desc="Adding to FAISS"):
    end = min(start + ADD_BATCH_SIZE, embeddings.shape[0])

    vecs = np.asarray(embeddings[start:end])
    ids = np.asarray(entity_ids[start:end])

    if vecs.dtype != np.float32:
        raise RuntimeError(
            f"FAISS IndexFlatIP requires float32 vectors, but embeddings dtype is {vecs.dtype}. "
            "No automatic conversion is done."
        )

    if ids.dtype != np.int64:
        raise RuntimeError(
            f"FAISS IDs must be int64, but entity_ids dtype is {ids.dtype}."
        )

    index.add_with_ids(vecs, ids)

if index.ntotal != embeddings.shape[0]:
    raise RuntimeError(f"FAISS ntotal mismatch: {index.ntotal} != {embeddings.shape[0]}")

faiss.write_index(index, str(ENTITY_FAISS_LOCAL))

meta = {
    "dataset": DATASET,
    "model_name": MODEL_NAME,
    "model_load_dtype": "default",
    "embedding_file_dtype": str(embeddings.dtype),
    "embedding_dim": int(dim),
    "num_entities": int(index.ntotal),
    "faiss_index_type": "IndexIDMap2(IndexFlatIP)",
    "similarity": "cosine_similarity_via_inner_product_on_l2_normalized_vectors",
    "entity_catalog_jsonl": ENTITY_CATALOG_JSONL_LOCAL.name,
    "entity_embeddings_npy": ENTITY_EMBEDDINGS_NPY_LOCAL.name,
    "entity_ids_npy": ENTITY_IDS_NPY_LOCAL.name,
    "faiss_index": ENTITY_FAISS_LOCAL.name,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(meta, ENTITY_META_JSON_LOCAL)

print("Saved FAISS index:", ENTITY_FAISS_LOCAL)
print("Saved meta:", ENTITY_META_JSON_LOCAL)
print("FAISS ntotal:", index.ntotal)
print("FAISS dim:", index.d)

Adding to FAISS:   0%|          | 0/7 [00:00<?, ?it/s]

Saved FAISS index: /content/kg_construct_work/hotpotqa/hotpotqa_entities.indexflatip.idmap.faiss
Saved meta: /content/kg_construct_work/hotpotqa/hotpotqa_entity_index_meta.json
FAISS ntotal: 327873
FAISS dim: 4096


In [11]:
# Verify FAISS search and entity_id mapping.
import json
import numpy as np
import faiss

# Reload index from disk.
index = faiss.read_index(str(ENTITY_FAISS_LOCAL))

# Build id -> entity map from JSONL.
id_to_entity = {}

with open(ENTITY_CATALOG_JSONL_LOCAL, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        id_to_entity[int(rec["entity_id"])] = rec["entity"]

def search_entities(query_texts, top_k=5):
    # Search normalized entity index.
    query_norm = [normalize_entity_text(q) for q in query_texts]

    query_embeddings = model.encode(
    query_norm,
    batch_size=len(query_norm),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
    )

    query_embeddings = np.asarray(query_embeddings)

    if query_embeddings.dtype != np.float32:
        raise RuntimeError(
            f"FAISS search requires float32 query vectors, but query dtype is {query_embeddings.dtype}. "
            "No automatic conversion is done."
        )

    scores, ids = index.search(query_embeddings, top_k)

    results = []

    for q, q_scores, q_ids in zip(query_texts, scores, ids):
        rows = []

        for score, entity_id in zip(q_scores, q_ids):
            if int(entity_id) == -1:
                continue

            rows.append({
                "entity_id": int(entity_id),
                "entity": id_to_entity.get(int(entity_id)),
                "score": float(score),
            })

        results.append({
            "query": q,
            "normalized_query": normalize_entity_text(q),
            "results": rows,
        })

    return results

test_queries = [
    "German language",
    "Emily Jewell",
    "United States",
]

test_results = search_entities(test_queries, top_k=5)

print(json.dumps(test_results, ensure_ascii=False, indent=2))

[
  {
    "query": "German language",
    "normalized_query": "german language",
    "results": [
      {
        "entity_id": 63882,
        "entity": "german language",
        "score": 0.9989035129547119
      },
      {
        "entity_id": 834,
        "entity": "german",
        "score": 0.953589677810669
      },
      {
        "entity_id": 63833,
        "entity": "deutsch",
        "score": 0.9314674735069275
      },
      {
        "entity_id": 149316,
        "entity": "deutch",
        "score": 0.915795087814331
      },
      {
        "entity_id": 63974,
        "entity": "german sprachraum",
        "score": 0.8854782581329346
      }
    ]
  },
  {
    "query": "Emily Jewell",
    "normalized_query": "emily jewell",
    "results": [
      {
        "entity_id": 109415,
        "entity": "emily jewell",
        "score": 1.0017927885055542
      },
      {
        "entity_id": 323412,
        "entity": "emily bailes",
        "score": 0.9339562654495239
      },
      {

In [12]:
# Copy final outputs from local disk to Google Drive.
import shutil
from pathlib import Path

local_outputs = [
    NORMALIZED_KG_JSONL_LOCAL,
    ENTITY_CATALOG_JSONL_LOCAL,
    ENTITY_EMBEDDINGS_NPY_LOCAL,
    ENTITY_IDS_NPY_LOCAL,
    ENTITY_FAISS_LOCAL,
    ENTITY_META_JSON_LOCAL,
    AUDIT_JSON_LOCAL,
]

copied_paths = []

for src in local_outputs:
    src = Path(src)

    if not src.exists():
        raise FileNotFoundError(f"Missing local output: {src}")

    dst = CONSTRUCT_DIR / src.name
    shutil.copy2(src, dst)
    copied_paths.append(dst)

print("Copied outputs to Drive:")
for path in copied_paths:
    print(path)

Copied outputs to Drive:
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_kg_extractions.clean.normalized_entities.jsonl
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entities.jsonl
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_embeddings.float32.npy
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_ids.int64.npy
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entities.indexflatip.idmap.faiss
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_index_meta.json
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_normalization_audit.json


In [13]:
# Final output verification.
from pathlib import Path
import json
import faiss
import numpy as np

expected_drive_outputs = {
    "normalized_kg_jsonl": CONSTRUCT_DIR / NORMALIZED_KG_JSONL_LOCAL.name,
    "entity_catalog_jsonl": CONSTRUCT_DIR / ENTITY_CATALOG_JSONL_LOCAL.name,
    "entity_embeddings_npy": CONSTRUCT_DIR / ENTITY_EMBEDDINGS_NPY_LOCAL.name,
    "entity_ids_npy": CONSTRUCT_DIR / ENTITY_IDS_NPY_LOCAL.name,
    "entity_faiss": CONSTRUCT_DIR / ENTITY_FAISS_LOCAL.name,
    "entity_meta_json": CONSTRUCT_DIR / ENTITY_META_JSON_LOCAL.name,
    "audit_json": CONSTRUCT_DIR / AUDIT_JSON_LOCAL.name,
}

for name, path in expected_drive_outputs.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")

    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{name}: {path} | {size_mb:.2f} MB")

drive_index = faiss.read_index(str(expected_drive_outputs["entity_faiss"]))
drive_embeddings = np.load(expected_drive_outputs["entity_embeddings_npy"], mmap_mode="r")
drive_entity_ids = np.load(expected_drive_outputs["entity_ids_npy"])

with open(expected_drive_outputs["entity_meta_json"], "r", encoding="utf-8") as f:
    drive_meta = json.load(f)

print("\nFinal checks:")
print("FAISS ntotal:", drive_index.ntotal)
print("Embeddings shape:", drive_embeddings.shape)
print("Entity IDs shape:", drive_entity_ids.shape)
print("Meta:")
print(json.dumps(drive_meta, ensure_ascii=False, indent=2))

if drive_index.ntotal != drive_embeddings.shape[0]:
    raise RuntimeError("Final FAISS index and embeddings row count mismatch.")

if drive_embeddings.shape[0] != drive_entity_ids.shape[0]:
    raise RuntimeError("Final embeddings and entity IDs row count mismatch.")

print("\nAll final outputs are valid.")

normalized_kg_jsonl: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_kg_extractions.clean.normalized_entities.jsonl | 244.00 MB
entity_catalog_jsonl: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entities.jsonl | 80.62 MB
entity_embeddings_npy: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_embeddings.float32.npy | 5123.02 MB
entity_ids_npy: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_ids.int64.npy | 2.50 MB
entity_faiss: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entities.indexflatip.idmap.faiss | 5125.52 MB
entity_meta_json: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_index_meta.json | 0.00 MB
audit_json: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_normalization_audit.json | 0.00 MB

Final checks:
FAISS ntotal: 327873
Embeddings s